# BRISC 2025 Baseline — Attention Residual U-Net

**Target:** Tumour Dice ≥ 0.80 (BRISC is harder than CAMUS because tumours are small ~1.7% of image and vary widely in shape).

## Kaggle setup
1. Add datasets: `briscdataset/brisc2025`  +  `iteris-pkg`
2. Accelerator: GPU T4 x2
3. Persistence: Files only
4. Save Version → Save & Run All (Commit)

**Path auto-detection** lives in Cell 1 — if your BRISC dataset is mounted somewhere unusual, edit `BRISC_ROOT_OVERRIDE` to a hardcoded path.

## 0 · Install (no kernel restart needed)

In [ ]:
!pip install -r /kaggle/input/datasets/anfaalhossain/iteris-pkg/requirements.txt --quiet --upgrade
print('✓ Setup complete. Run All from cell 1 onwards.')

## 1 · Load config + locate the BRISC dataset

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '/kaggle/input/datasets/anfaalhossain/iteris-pkg')

from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config('/kaggle/input/datasets/anfaalhossain/iteris-pkg/configs/brisc.yaml')

# Locate BRISC under /kaggle/input — handles any Kaggle wrapper layout.
# If auto-detection fails, set BRISC_ROOT_OVERRIDE to the correct path.
BRISC_ROOT_OVERRIDE = None       # e.g. '/kaggle/input/brisc2025'
if BRISC_ROOT_OVERRIDE:
    cfg['data_root'] = BRISC_ROOT_OVERRIDE
else:
    candidates = [p for p in Path('/kaggle/input').rglob('brisc*') if p.is_dir()]
    if not candidates:
        raise RuntimeError('No BRISC folder found. Attach the dataset or set BRISC_ROOT_OVERRIDE.')
    # Pick the shallowest match (most likely the dataset root)
    cfg['data_root'] = str(min(candidates, key=lambda p: len(p.parts)))

cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Dataset      : {cfg["dataset"]}')
print(f'Data root    : {cfg["data_root"]}')
print(f'Image size   : {cfg["image_size"]}  in_channels={cfg["in_channels"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Normalisation: {cfg["normalize"]}   binarise labels: {cfg["binarize_labels"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}')
print(f'Device       : {get_device()}')

## 2 · Train

In [ ]:
from iteris.training import run_training

result      = run_training(cfg)
model       = result['model']
history     = result['history']
best_dice   = result['best_dice']
test_loader = result['test_loader']
checkpoint  = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.80)

## 4 · Test-set evaluation

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 5 · Export predicted masks (init state for DRL agents)

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 6 · Qualitative grid

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=4)

## 7 · Summary JSON

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)